# New Deutsch - Jozsa — IQM edition

This notebook runs on **IQM** quantum computers — **Garnet** (20 qubits) or **Emerald** (54 qubits) — via **IQM Resonance**.

IQM chips use the native gate set **PRX (phased-RX) + CZ**. The helper `run_counts` transpiles each circuit to these native gates before running, so the circuit-construction code below stays unchanged.

**Pick your backend by editing the single `TARGET` line in the next cell.** Options:

- `"fake_apollo"` — local 20-qubit noisy simulator, **no token** needed (default, fully offline).
- `"garnet:mock"` / `"emerald:mock"` — IQM Resonance **cloud mock** (needs a token, **no QPU time**). It validates the job pipeline (token, transpilation, connectivity, instruction limit) but returns **random bitstrings**, not simulated physics: verify logic with `fake_apollo`.
- `"garnet"` / `"emerald"` — the **real QPU** (needs a token **and** QPU credits).

See `README.md` in this folder for setup and token instructions.

In [ ]:
from iqm_backend import get_backend, run_counts   # helper module in this folder (see iqm_backend.py)
# run_counts(circuit, TARGET, shots=...) = transpile to the IQM native gates (PRX + CZ) + run + get_counts()
# get_backend(TARGET) returns the raw Qiskit backend object if you want to call backend.run yourself
TARGET = "fake_apollo"   # change this one line to choose the backend (see the table above)


In [ ]:
from qiskit import QuantumCircuit
from qiskit.visualization import plot_histogram
import numpy as np

In [ ]:
def create_oracle(n_qubits, function_type='constant', constant_value=0):
    """
    Create an oracle for the Deutsch-Jozsa algorithm.
    
    Args:
        n_qubits (int): Number of input qubits
        function_type (str): 'constant' or 'balanced'
        constant_value (int): 0 or 1, used if function_type is 'constant'
    
    Returns:
        QuantumCircuit: Oracle circuit
    """
    oracle = QuantumCircuit(n_qubits + 1)  # +1 for auxiliary qubit
    
    if function_type == 'constant':
        if constant_value == 1:
            oracle.x(n_qubits)  # Apply X gate to auxiliary qubit for f(x)=1
    else:  # balanced
        # Create a balanced function by applying CNOT gates to half of the input qubits
        for i in range(n_qubits // 2):
            oracle.cx(i, n_qubits)
    return oracle

In [ ]:
def deutsch_jozsa_circuit(n_qubits, oracle):
    """
    Create the complete Deutsch-Jozsa circuit.
    
    Args:
        n_qubits (int): Number of input qubits
        oracle (QuantumCircuit): Oracle circuit
    
    Returns:
        QuantumCircuit: Complete circuit
    """
    # Create quantum circuit with classical bits for measurement
    circuit = QuantumCircuit(n_qubits + 1, n_qubits)
    
    # Initialize auxiliary qubit to |1⟩
    circuit.x(n_qubits)
    
    # Apply Hadamard gates to all qubits
    for i in range(n_qubits + 1):
        circuit.h(i)
    
    # Apply oracle
    circuit.compose(oracle, inplace=True)
    
    # Apply Hadamard gates to input qubits
    for i in range(n_qubits):
        circuit.h(i)
    
    # Measure input qubits
    circuit.measure(range(n_qubits), range(n_qubits))
    
    return circuit

In [ ]:
# Parameters
n_qubits = 3  # Number of input qubits

# Create circuits with constant and balanced oracles
constant_oracle = create_oracle(n_qubits, 'constant', 0)
constant_circuit = deutsch_jozsa_circuit(n_qubits, constant_oracle)
constant_circuit.draw('mpl')

In [ ]:
balanced_oracle = create_oracle(n_qubits, 'balanced')
balanced_circuit = deutsch_jozsa_circuit(n_qubits, balanced_oracle)
balanced_circuit.draw('mpl')

In [ ]:
# Run both circuits on the selected IQM backend for comparison
print("IQM Results:")
constant_sim_counts = run_counts(constant_circuit, TARGET, shots=2048)
balanced_sim_counts = run_counts(balanced_circuit, TARGET, shots=2048)
print("Constant function:", constant_sim_counts)
print("Balanced function:", balanced_sim_counts)

In [ ]:
# Transpile both circuits to the IQM native gate set (PRX + CZ) and run them.
# run_counts(..., return_transpiled=True) returns (counts, transpiled_circuit).
constant_counts, transpiled_constant_circuit = run_counts(
    constant_circuit, TARGET, shots=2048, return_transpiled=True)
balanced_counts, transpiled_balanced_circuit = run_counts(
    balanced_circuit, TARGET, shots=2048, return_transpiled=True)

print("\nIQM Quantum Results:")
print("Constant function:", constant_counts)
print("Balanced function:", balanced_counts)

In [ ]:
print(f"Original constant circuit depth: {constant_circuit.depth()}")
print(f"Transpiled constant circuit depth: {transpiled_constant_circuit.depth()}")
print(f"Original balanced circuit depth: {balanced_circuit.depth()}")
print(f"Transpiled balanced circuit depth: {transpiled_balanced_circuit.depth()}")

In [ ]:
balanced_circuit.draw("mpl")

In [ ]:
transpiled_balanced_circuit.draw("mpl")

In [ ]:
def interpret_results(counts):
    # If (nearly) all shots land on the all-zeros string the function is
    # constant; otherwise it is balanced.
    all_zeros = '0' * n_qubits
    print(all_zeros)
    total = sum(counts.values())
    fraction_zeros = counts.get(all_zeros, 0) / total if total else 0
    is_constant = fraction_zeros > 0.9
    return "Constant" if is_constant else "Balanced"

In [ ]:
print("\nInterpretation of IBM Quantum results:")
print("Constant function:", interpret_results(constant_counts))
print("Balanced function:", interpret_results(balanced_counts))